# Telco Customer Churn - Exploratory Data Analysis

This notebook explores the IBM Telco Customer Churn dataset to understand the data distribution, identify patterns, and inform feature engineering and modeling decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load data
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## Basic Information

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicated rows: {df.duplicated().sum()}")

## Target Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
churn_counts.plot(kind='bar', ax=ax[0], color=['skyblue', 'salmon'])
ax[0].set_title('Churn Count')
ax[0].set_ylabel('Count')

churn_pct.plot(kind='bar', ax=ax[1], color=['skyblue', 'salmon'])
ax[1].set_title('Churn Percentage')
ax[1].set_ylabel('Percentage')

plt.tight_layout()
plt.show()

print(f"Churn rate: {churn_pct['Yes']:.1f}%")

## Numeric Features

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[numeric_cols].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, col in enumerate(numeric_cols):
    df[col] = pd.to_numeric(df[col], errors='coerce')
    for churn in df['Churn'].unique():
        subset = df[df['Churn'] == churn][col].dropna()
        axes[i].hist(subset, alpha=0.6, label=f'Churn={churn}', bins=30)
    axes[i].set_title(f'{col} Distribution by Churn')
    axes[i].set_xlabel(col)
    axes[i].legend()
plt.tight_layout()
plt.show()

## Categorical Features

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['customerID', 'TotalCharges']]

n_cols = 3
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    crosstab = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    crosstab.plot(kind='bar', ax=axes[i], color=['skyblue', 'salmon'], stacked=False)
    axes[i].set_title(f'{col} vs Churn')
    axes[i].set_ylabel('Percentage')
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
# Encode binary for correlation
df_corr = df.copy()
df_corr['Churn'] = df_corr['Churn'].map({'No': 0, 'Yes': 1})
df_corr['gender'] = df_corr['gender'].map({'Female': 0, 'Male': 1})
for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df_corr[col] = df_corr[col].map({'No': 0, 'Yes': 1})

# Select numeric columns
corr_cols = numeric_cols + ['SeniorCitizen', 'gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
corr_matrix = df_corr[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Key Insights

1. **Churn Rate**: Approximately 26.5% of customers churn.
2. **Tenure**: Customers with lower tenure are more likely to churn.
3. **Contract Type**: Month-to-month contracts have significantly higher churn.
4. **Payment Method**: Electronic check users churn more frequently.
5. **Services**: Customers without online security/tech support show higher churn.

These insights will guide feature engineering and modeling decisions.